# Audio Segmentation Pipeline

This Colab provides a streamlined pipeline for extracting labeled audio segments from Google Cloud Storage (GCS) using JSONL annotations and exporting them back to GCS.

### Core Functions
1. **Manifest Retrieval**: Fetches target file lists from GCS. Only annotated files are processed.
2. **Local Audio Caching**: Downloads source audio into a local cache. On subsequent runs, it skips downloads if the file is already present locally.
3. **Ground Truth Slicing**: Uses `librosa` and `soundfile` to precisely extract audio segments as defined by the start/end timestamps in the manifest.
4. **Automated Export**:
    * Uploads processed FLAC segments to GCS organized by `example_id`.
    * Generates and uploads a `batch_manifest.jsonl` containing metadata for all segments (GCS paths, offsets, and durations) to facilitate downstream ASR evaluation.

wd-transcription-data/manifests/one_hour_pilot_transcriptions.json\

In [ ]:
# @title Install dependencies
!pip install -q --upgrade \
    loguru \
    librosa \
    soundfile

In [ ]:
# @title Imports and environment configuration
import json
import sys
from pathlib import Path
from urllib.parse import urlparse

import librosa
import soundfile as sf
from google.cloud import storage
from google.colab import auth
from loguru import logger

GCP_PROJECT_ID = ""  # @param {type:"string"}
GCS_BUCKET = ""  # @param {type:"string"}
SOURCE_MANIFEST_URI = (
    f"gs://{GCS_BUCKET}/manifests/one_hour_pilot_transcriptions.json"
)
GCS_OUTPUT_PREFIX = "segmented_audio/one_hour_pilot_audio"

LOCAL_BASE_PATH = "/content"
CACHE_DIR = f"{LOCAL_BASE_PATH}/raw_audio"
SEGMENTS_DIR = f"{LOCAL_BASE_PATH}/segments"
BATCH_MANIFEST_FILENAME = "batch_manifest.jsonl"

# Audio specs
SAMPLE_RATE = 16000
CHANNELS = 1

# Initialize loguru
logger.remove()
logger.add(sys.stderr, format="<level>{level}</level>: {message}");

In [ ]:
# @title Authentication and client initialization
auth.authenticate_user()
!gcloud config set project {GCP_PROJECT_ID} --quiet

# Initialize GCS Client
gcs_client = storage.Client(project=GCP_PROJECT_ID)

In [ ]:
# @title Helper functions
def ensure_local_gcs_audio(gcs_uri: str) -> str:
    """Ensures the audio file from GCS is available locally in CACHE_DIR."""
    parsed = urlparse(gcs_uri)
    bucket_name = parsed.netloc
    blob_name = parsed.path.lstrip("/")

    filename = Path(blob_name).name
    local_path = Path(CACHE_DIR) / filename

    if not local_path.exists():
        logger.info(f"Downloading {filename} from GCS...")
        bucket = gcs_client.bucket(bucket_name)
        blob = bucket.blob(blob_name)
        blob.download_to_filename(str(local_path))
    return str(local_path)


def cleanup_gcs_output() -> None:
    """Deletes existing blobs in the output directory for a clean run."""
    bucket = gcs_client.bucket(GCS_BUCKET)
    blobs = list(bucket.list_blobs(prefix=GCS_OUTPUT_PREFIX))
    if blobs:
        logger.info(
            f"Cleaning existing files from gs://{GCS_BUCKET}/{GCS_OUTPUT_PREFIX}..."
        )
        bucket.delete_blobs(blobs)


def run_pilot_pipeline() -> None:
    """Orchestrates extraction from the JSONL manifest in GCS."""
    Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
    Path(SEGMENTS_DIR).mkdir(parents=True, exist_ok=True)
    cleanup_gcs_output()

    # 1. Load the JSONL manifest from GCS
    parsed_manifest = urlparse(SOURCE_MANIFEST_URI)
    m_bucket = gcs_client.bucket(parsed_manifest.netloc)
    m_blob = m_bucket.blob(parsed_manifest.path.lstrip("/"))
    content = m_blob.download_as_text()

    # Group segments by audio file to minimize downloads/re-loads
    manifest_data = [json.loads(line) for line in content.strip().split("\n")]
    files_to_process = {}
    for entry in manifest_data:
        path = entry["audio_filepath"]
        if path not in files_to_process:
            files_to_process[path] = []
        files_to_process[path].append(entry)

    output_bucket = gcs_client.bucket(GCS_BUCKET)
    final_manifest_entries = []

    # 2. Process each audio file
    for gcs_audio_path, segments in files_to_process.items():
        local_src_path = ensure_local_gcs_audio(gcs_audio_path)
        example_id = Path(gcs_audio_path).stem

        logger.info(f"Processing {len(segments)} segments for: {example_id}")
        y, sr = librosa.load(local_src_path, sr=SAMPLE_RATE)

        for i, seg in enumerate(segments):
            start_s = seg["offset"]
            duration_s = seg["duration"]
            end_s = start_s + duration_s

            # Slice
            y_slice = y[int(start_s * sr) : int(end_s * sr)]

            seg_id = f"{i:03d}"
            filename = f"{example_id}__seg{seg_id}.flac"
            local_slice_path = Path(SEGMENTS_DIR) / filename
            sf.write(str(local_slice_path), y_slice, sr)

            # Upload
            blob_name = f"{GCS_OUTPUT_PREFIX}/{example_id}/{filename}"
            blob = output_bucket.blob(blob_name)
            blob.upload_from_filename(str(local_slice_path))

            # Metadata entry
            final_manifest_entries.append(
                {
                    "audio_filepath": f"gs://{GCS_BUCKET}/{blob_name}",
                    "example_id": example_id,
                    "offset": start_s,
                    "duration": duration_s,
                    "segment_id": seg_id,
                    "text": seg.get("text", ""),
                }
            )

    # 3. Create and upload the final batch manifest
    local_manifest = Path(LOCAL_BASE_PATH) / BATCH_MANIFEST_FILENAME
    with open(local_manifest, "w") as f:
        for entry in final_manifest_entries:
            f.write(json.dumps(entry) + "\n")

    gcs_manifest_path = f"{GCS_OUTPUT_PREFIX}/{BATCH_MANIFEST_FILENAME}"
    output_bucket.blob(gcs_manifest_path).upload_from_filename(
        str(local_manifest)
    )

    logger.info(
        f"Pipeline Complete. {len(final_manifest_entries)} segments uploaded to gs://{GCS_BUCKET}/{GCS_OUTPUT_PREFIX}"
    )

In [ ]:
# @title Create the pilot segments and manifest file
run_pilot_pipeline()